In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import *
import itertools

In [2]:
# Load dataset
ds = load_jsonl_file('./../datasets/MATE-dev/mm_0shot_llava_hfllava_1.5_7b_hf.jsonl')
mate_df = pd.read_csv('./data/mate_df.csv', index_col=0)


5500it [00:00, 6999.36it/s] 


In [3]:
mate_df

,idx,image,object_count,color,shape,material,color_shape,color_material,shape_material,color_shape_material,sizes
0,0,7e8a39cbce31181466491620a63de934.png,9,"['gray', 'red', 'gray', 'cyan', 'brown', 'gray...","['cylinder', 'cylinder', 'cylinder', 'cube', '...","['metal', 'rubber', 'rubber', 'metal', 'metal'...","['gray_cylinder', 'red_cylinder', 'gray_cylind...","['gray_metal', 'red_rubber', 'gray_rubber', 'c...","['cylinder_metal', 'cylinder_rubber', 'cylinde...","['gray_cylinder_metal', 'red_cylinder_rubber',...","[0.351, 0.351, 0.701, 0.7, 0.7, 0.351, 0.351, ..."
1,1,4b69b6eb323f926d21cd7b4d3aa6cf94.png,10,"['red', 'brown', 'blue', 'blue', 'cyan', 'gray...","['cylinder', 'cylinder', 'cylinder', 'sphere',...","['rubber', 'rubber', 'metal', 'metal', 'metal'...","['red_cylinder', 'brown_cylinder', 'blue_cylin...","['red_rubber', 'brown_rubber', 'blue_metal', '...","['cylinder_rubber', 'cylinder_rubber', 'cylind...","['red_cylinder_rubber', 'brown_cylinder_rubber...","[0.351, 0.701, 0.351, 0.351, 0.351, 0.351, 0.3..."
2,2,60b990927b64079fa61f9fb9591d6994.png,6,"['cyan', 'brown', 'gray', 'green', 'purple', '...","['cube', 'sphere', 'cube', 'cube', 'cube', 'cy...","['metal', 'metal', 'rubber', 'metal', 'rubber'...","['cyan_cube', 'brown_sphere', 'gray_cube', 'gr...","['cyan_metal', 'brown_metal', 'gray_rubber', '...","['cube_metal', 'sphere_metal', 'cube_rubber', ...","['cyan_cube_metal', 'brown_sphere_metal', 'gra...","[0.7, 0.701, 0.701, 0.351, 0.351, 0.701]"
3,3,ddfc53034f0a421970051e98c9dc6139.png,6,"['yellow', 'purple', 'cyan', 'gray', 'gray', '...","['cylinder', 'cone', 'cylinder', 'cylinder', '...","['rubber', 'metal', 'metal', 'rubber', 'rubber...","['yellow_cylinder', 'purple_cone', 'cyan_cylin...","['yellow_rubber', 'purple_metal', 'cyan_metal'...","['cylinder_rubber', 'cone_metal', 'cylinder_me...","['yellow_cylinder_rubber', 'purple_cone_metal'...","[0.701, 0.7, 0.701, 0.7, 0.351, 0.351]"
4,4,ee3ed5c55c080eeed55ca2b2a24d7285.png,7,"['brown', 'green', 'cyan', 'blue', 'yellow', '...","['cone', 'cube', 'cylinder', 'cylinder', 'cyli...","['metal', 'metal', 'metal', 'rubber', 'metal',...","['brown_cone', 'green_cube', 'cyan_cylinder', ...","['brown_metal', 'green_metal', 'cyan_metal', '...","['cone_metal', 'cube_metal', 'cylinder_metal',...","['brown_cone_metal', 'green_cube_metal', 'cyan...","[0.7, 0.7, 0.7, 0.701, 0.701, 0.7, 0.351]"
...,...,...,...,...,...,...,...,...,...,...,...
5495,5495,a7386765d625fbdcdb1b99eee7daaf51.png,3,"['cyan', 'red', 'green']","['cone', 'sphere', 'cone']","['metal', 'metal', 'metal']","['cyan_cone', 'red_sphere', 'green_cone']","['cyan_metal', 'red_metal', 'green_metal']","['cone_metal', 'sphere_metal', 'cone_metal']","['cyan_cone_metal', 'red_sphere_metal', 'green...","[0.351, 0.7, 0.7]"
5496,5496,271618ad987006f91d6e4a0cb66ac0b1.png,8,"['blue', 'green', 'purple', 'brown', 'gray', '...","['cube', 'sphere', 'cube', 'cylinder', 'sphere...","['rubber', 'metal', 'metal', 'metal', 'rubber'...","['blue_cube', 'green_sphere', 'purple_cube', '...","['blue_rubber', 'green_metal', 'purple_metal',...","['cube_rubber', 'sphere_metal', 'cube_metal', ...","['blue_cube_rubber', 'green_sphere_metal', 'pu...","[0.7, 0.7, 0.351, 0.351, 0.7, 0.351, 0.701, 0...."
5497,5497,51c762d0d4421ba2249f9655daf6d016.png,8,"['red', 'blue', 'brown', 'yellow', 'green', 'g...","['cylinder', 'sphere', 'sphere', 'cube', 'sphe...","['metal', 'rubber', 'metal', 'metal', 'metal',...","['red_cylinder', 'blue_sphere', 'brown_sphere'...","['red_metal', 'blue_rubber', 'brown_metal', 'y...","['cylinder_metal', 'sphere_rubber', 'sphere_me...","['red_cylinder_metal', 'blue_sphere_rubber', '...","[0.351, 0.351, 0.351, 0.701, 0.351, 0.7, 0.351..."
5498,5498,ec795e2594e154873f2c53b519fbb728.png,4,"['brown', 'blue', 'green', 'cyan']","['cylinder', 'cube', 'sphere', 'cylinder']","['rubber', 'metal', 'rubber', 'metal']","['brown_cylinder', 'blue_cube', 'green_sphere'...","['brown_rubber', 'blue_metal', 'green_rubber',...","['cylinder_rubber', 

In [ ]:
create_mate_df = False
if create_mate_df:
    mate_df = pd.DataFrame(
        {'idx': list(range(0,len(ds))),
        'image': [data['image'] for data in ds],
        'object_count': [data['object_count'] for data in ds],
        'color': [[obj['color'] for obj in data['scene']['objects']] for data in ds],
        'shape': [[obj['shape'] for obj in data['scene']['objects']] for data in ds],
        'material': [[obj['material'] for obj in data['scene']['objects']] for data in ds],
        'color_shape': [[f"{obj['color']}_{obj['shape']}" for obj in data['scene']['objects']] for data in ds],
        'color_material': [[f"{obj['color']}_{obj['material']}" for obj in data['scene']['objects']] for data in ds],
        'shape_material': [[f"{obj['shape']}_{obj['material']}" for obj in data['scene']['objects']] for data in ds],
        'color_shape_material': [[f"{obj['color']}_{obj['shape']}_{obj['material']}" for obj in data['scene']['objects']] for data in ds],
        'size': [[obj['size'] for obj in data['scene']['objects']] for data in ds]}
    )


    mate_df.to_csv('./data/mate_df.csv')

In [63]:
# sizes of object
object_size_ls = set([object_size for sizes in mate_df['sizes'].to_list() for object_size in sizes])
object_size_ls

{0.35, 0.351, 0.7, 0.701}

In [13]:

# Number of objects
nobjects = list(range(0,12))

for i in nobjects:
    # Filter dataset for a specific number of objects 
    ds_ = [1 for data in ds if data['object_count'] == i]
    print(f"The number of examples with {i} objects is {len(ds_)}")

The number of examples with 0 objects is 0
The number of examples with 1 objects is 0
The number of examples with 2 objects is 0
The number of examples with 3 objects is 670
The number of examples with 4 objects is 696
The number of examples with 5 objects is 681
The number of examples with 6 objects is 674
The number of examples with 7 objects is 688
The number of examples with 8 objects is 746
The number of examples with 9 objects is 681
The number of examples with 10 objects is 664
The number of examples with 11 objects is 0


In [ ]:


task_dict = {
    'color': ['gray', 'yellow', 'red', 'blue', 'green'],
    'material': ['rubber', 'metal'],
    'shape': ['cone', 'cylinder', 'cube']
}

combinations = list(itertools.product(task_dict['color'], task_dict['shape'], task_dict['material']))
combinations

[('gray', 'cone', 'rubber'),
 ('gray', 'cone', 'metal'),
 ('gray', 'cylinder', 'rubber'),
 ('gray', 'cylinder', 'metal'),
 ('gray', 'cube', 'rubber'),
 ('gray', 'cube', 'metal'),
 ('yellow', 'cone', 'rubber'),
 ('yellow', 'cone', 'metal'),
 ('yellow', 'cylinder', 'rubber'),
 ('yellow', 'cylinder', 'metal'),
 ('yellow', 'cube', 'rubber'),
 ('yellow', 'cube', 'metal'),
 ('red', 'cone', 'rubber'),
 ('red', 'cone', 'metal'),
 ('red', 'cylinder', 'rubber'),
 ('red', 'cylinder', 'metal'),
 ('red', 'cube', 'rubber'),
 ('red', 'cube', 'metal'),
 ('blue', 'cone', 'rubber'),
 ('blue', 'cone', 'metal'),
 ('blue', 'cylinder', 'rubber'),
 ('blue', 'cylinder', 'metal'),
 ('blue', 'cube', 'rubber'),
 ('blue', 'cube', 'metal'),
 ('green', 'cone', 'rubber'),
 ('green', 'cone', 'metal'),
 ('green', 'cylinder', 'rubber'),
 ('green', 'cylinder', 'metal'),
 ('green', 'cube', 'rubber'),
 ('green', 'cube', 'metal')]

In [77]:
combinations = list(itertools.product(task_dict['shape']))
combinations


for combination in combinations:
        nsamples = len(mate_df[(mate_df['object_count'] == 7) & (mate_df['shape'].apply(lambda x: '_'.join(combination) in x))])
        if nsamples>99:
                print(f"The number of samples with {combination} objects: {nsamples}")

The number of samples with ('cone',) objects: 438
The number of samples with ('cylinder',) objects: 666
The number of samples with ('cube',) objects: 663


In [69]:
for nobj in [3,7,10]:
        for object_size in object_size_ls:
                print(f"object size {object_size}, nobjects {nobj}, nsamples {mate_df[(mate_df['object_count'] == nobj)&(mate_df['sizes'].apply(lambda x: object_size in x))].shape[0]}")

object size 0.351, nobjects 3, nsamples 594
object size 0.701, nobjects 3, nsamples 235
object size 0.35, nobjects 3, nsamples 9
object size 0.7, nobjects 3, nsamples 491
object size 0.351, nobjects 7, nsamples 681
object size 0.701, nobjects 7, nsamples 426
object size 0.35, nobjects 7, nsamples 17
object size 0.7, nobjects 7, nsamples 656
object size 0.351, nobjects 10, nsamples 664
object size 0.701, nobjects 10, nsamples 470
object size 0.35, nobjects 10, nsamples 44
object size 0.7, nobjects 10, nsamples 650
